# Tüdőrák predikciója vizelet LC–MS metabolomikai profilból

## feature selection és gépi tanulási modellek összehasonlítása

### Ambrus Csaba


## 0. Setup

In [ ]:
# Init gdrive and python environment

from google.colab import drive
drive.mount('/content/drive')

%cd /content

import os
if not os.path.exists("/content/MetabolKD/.git"):
    !git clone https://github.com/csambrus/MetabolKD.git
else:
    %cd /content/MetabolKD
    !git fetch origin
    !git pull origin main

%cd /content/MetabolKD
!pip install -r ./requirements.txt

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
fatal: destination path 'MetabolKD' already exists and is not an empty directory.
/content/MetabolKD


In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = "content/MetabolKD"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)
from src.runtime_setup import setup_tensorflow_runtime, setup_notebook_error_logger, set_global_seed

set_global_seed()
setup_notebook_error_logger()
setup_tensorflow_runtime()


PROJECT_ROOT: content/MetabolKD
Globalis seed beallitva: 42
✅ Globális hiba-logger aktív.
TF version: 2.20.0
GPUs:  []
CPU cores:  2
CUDA_VISIBLE_DEVICES = None
TF version = 2.20.0
Built with CUDA = True
GPUs = []
Logical GPUs = []
GPUs: []
No GPU visible to TensorFlow


2

## 1. Adatbetöltés


In [ ]:
from src.download_dataset import download_dataset, combine_pos_neg_to_feature_matrix

data = download_dataset()
X, meta = combine_pos_neg_to_feature_matrix(data)

# ML célváltozó
target_col = "class_label"  # Lung cancer vs Control

# Csak címkézett minták
mask = meta[target_col].notna()
X = X.loc[mask].copy()
meta = meta.loc[mask].copy()

# Biztonság: indexek egyezzenek
meta = meta.reindex(X.index)

X_raw = X
y = meta[target_col]

print("X_raw:", X_raw.shape)
print(y.value_counts(dropna=False))

ImportError: cannot import name 'combine_pos_neg_to_feature_matrix' from 'src.download_dataset' (/content/MetabolKD/src/download_dataset.py)

In [4]:
print(data["files"])
print(data["sample_tables"].keys())
print(data["assay_tables"].keys())
print(data["matrix_tables"].keys())

for name, df in data["matrix_tables"].items():
    print("\n---", name, df.shape)
    print(df.head())
    print(df.columns[:20].tolist())

['a_MTBLS28_NEG.txt', 'a_MTBLS28_POS.txt', 'i_Investigation.txt', 'm_MTBLS28_NEG_v2_maf.tsv', 'm_MTBLS28_POS_v2_maf.tsv', 's_MTBLS28.txt']
dict_keys(['s_MTBLS28.txt'])
dict_keys(['a_MTBLS28_NEG.txt', 'a_MTBLS28_POS.txt'])
dict_keys(['m_MTBLS28_NEG_v2_maf.tsv', 'm_MTBLS28_POS_v2_maf.tsv'])

--- m_MTBLS28_NEG_v2_maf.tsv (1359, 1026)
  database_identifier chemical_formula smiles inchi metabolite_identification  \
0                 NaN              NaN    NaN   NaN                       NaN   
1                 NaN              NaN    NaN   NaN                       NaN   
2                 NaN              NaN    NaN   NaN                       NaN   
3                 NaN              NaN    NaN   NaN                       NaN   
4                 NaN              NaN    NaN   NaN                       NaN   

  mass_to_charge fragmentation modifications charge retention_time  ...  \
0    59.01287386           NaN           NaN    NaN    71.42227731  ...   
1    61.98754668           NaN

## 2. QC (mintánkénti összjel és hiányzás)


In [ ]:
from src.metabolomics_qc import qc_sample_table
from src.metabolomics_plotting import plot_qc_bars

qc = qc_sample_table(X_raw)
print(qc.describe())
fig = plot_qc_bars(qc, meta, hue_col="progressor_status")
fig.savefig("qc_overview.png", dpi=150, bbox_inches="tight")


## 3. NMR-stílusú mátrixok (PQN, log1p, Pareto) + klasszikus LC–MS előfeldolgozás


In [ ]:
from src.metabolomics_preprocessing import nmr_style_matrices, preprocess_feature_matrix

blocks = nmr_style_matrices(X_raw, impute_first=True)
for k, df in blocks.items():
    print(k, df.shape)
X_pareto = blocks["X_pareto"]
X_log = blocks["X_log"]

X_proc, steps = preprocess_feature_matrix(X_raw)
print("Klasszikus LC–MS lépések:", " → ".join(steps))
X_proc.head()


## 4. PCA multipanel (Pareto + log1p térben, extra Z-score nélkül — mint az NMR `X_PARETO`)


In [ ]:
from src.metabolomics_multivariate import fit_pca
from src.metabolomics_plotting import plot_pca_multipanel, plot_pca_scores

pca_out = fit_pca(X_pareto, n_components=5, standardize=False)
scores = pca_out["scores"]
var = pca_out["explained_variance_ratio"]
print("Magyarázott variancia (első 5 PC):", [round(float(v), 4) for v in var])

fig = plot_pca_multipanel(pca_out, meta, hue_col="progressor_status")
fig.savefig("pca_multipanel.png", dpi=150, bbox_inches="tight")

fig2 = plot_pca_scores(
    scores, meta,
    pc_x="PC1", pc_y="PC2",
    hue_col="progressor_status",
    explained=var,
    title="PCA score (Pareto-skálázott log1p)",
)
fig2.savefig("pca_scores.png", dpi=150, bbox_inches="tight")


## 5. Univariáns elemzés + vulkán (Baseline: Progressor vs Non-progressor)


In [ ]:
# Csak baseline látogatás
bl = meta["visit"] == "Baseline"
Xb_log = X_log.loc[bl]
Xb_par = X_pareto.loc[bl]
metab = meta.loc[bl]

from src.metabolomics_univariate import differential_analysis
from src.metabolomics_plotting import plot_volcano, plot_volcano_categorized

diff = differential_analysis(
    Xb_log,
    metab["progressor_status"],
    group_a="Progressor",
    group_b="Non-progressor",
    include_cohen_d=True,
)
print(diff.head(15))
fig = plot_volcano(diff, alpha=0.05, fc_thresh=0.5)
fig.savefig("volcano_baseline_progressor.png", dpi=150, bbox_inches="tight")
figc = plot_volcano_categorized(
    diff,
    group_up="Progressor",
    group_down="Non-progressor",
    alpha=0.05,
    fc_thresh=0.5,
)
figc.savefig("volcano_categorized.png", dpi=150, bbox_inches="tight")


## 6. Top eltérések heatmap


In [ ]:
from src.metabolomics_plotting import plot_top_features_heatmap

top_n = 25
top_feats = diff.nsmallest(top_n, "padj")["feature"].tolist()
fig = plot_top_features_heatmap(
    Xb_par,
    top_feats,
    metab,
    group_col="progressor_status",
    max_samples=50,
)
fig.savefig("heatmap_top_features.png", dpi=150, bbox_inches="tight")


## 7. PLS-DA + VIP + S-plot (Pareto mátrix)


In [ ]:
from src.metabolomics_multivariate import fit_plsda_multiclass
from src.metabolomics_plotting import plot_s_plot_lv1
import matplotlib.pyplot as plt

plsda = fit_plsda_multiclass(
    Xb_par,
    metab["progressor_status"],
    n_components=3,
    scale=True,
)
s_pls = plsda["scores"]
vip = plsda["vip"]
print(vip.nlargest(10))

fig, ax = plt.subplots(figsize=(7, 5))
for lab in metab["progressor_status"].unique():
    m = metab["progressor_status"] == lab
    ax.scatter(s_pls.loc[m, "LV1"], s_pls.loc[m, "LV2"], label=str(lab), s=45, alpha=0.85, edgecolors="white", linewidths=0.3)
ax.set_xlabel("LV1")
ax.set_ylabel("LV2")
ax.set_title("PLS-DA score (Progressor címke)")
ax.legend(title="progressor_status")
ax.axhline(0, color="gray", lw=0.4)
ax.axvline(0, color="gray", lw=0.4)
fig.tight_layout()
fig.savefig("plsda_scores.png", dpi=150, bbox_inches="tight")

w1 = plsda["model"].pls.x_weights_[:, 0]
figs = plot_s_plot_lv1(Xb_par, metab["progressor_status"], "Progressor", w1)
figs.savefig("splot_lv1.png", dpi=150, bbox_inches="tight")


## 8. Random Forest — feature importance (felügyelt, gyors)


In [ ]:
from src.metabolomics_ml import random_forest_feature_importance

imp = random_forest_feature_importance(
    Xb_par,
    metab["progressor_status"],
    "Progressor",
    n_estimators=300,
)
print(imp.head(20))


## 9. Rövid értelmezés

- **QC**: extrém `total_signal` vagy `missing_frac` minták kiszűrhetők.
- **PCA**: fő irányok a legnagyobb közös variancia mentén; színezés a klinikai címkével.
- **Vulkán + FDR**: egyszerre nézünk hatást (log2FC) és bizonytalanságot (p / FDR).
- **PLS-DA + VIP + S-plot**: felügyelt irányok és jellemző-súlyok (NMR notebook mintájára).
- **RF importance**: nemlineáris alternatíva a fontos peak-ekhez.

A `NMR_Metabolomika_Pipeline_v2.ipynb` referenciához igazítva: **PQN / log1p / Pareto**, **multipanel PCA**, **FDR (statsmodels BH)**, **kategorizált vulkán**, **PLS-DA + VIP**; GPU/SHAP részek nélkül, LC–MS lipid adatra.

